# 004 · Priority Queues with heapq

> **Standard Library › Data Structures** · Python 3.14

`heapq` implements a binary **min-heap** — the data structure behind priority queues. Unlike .NET's `PriorityQueue<TElement, TPriority>`, it isn't a class: it's a set of functions that maintain the heap property on an ordinary `list`.

**In this notebook you will learn:**

- how `heappush` / `heappop` / `heapify` map to `PriorityQueue.Enqueue` / `Dequeue`
- `heappushpop` and `heapreplace` for efficient push-and-pop
- `nlargest` / `nsmallest` for top-N queries, and `merge` for sorted streams
- how to store `(priority, counter, item)` tuples to break ties safely
- max-heaps: the negation trick, and the new max-heap functions in Python 3.14

## .NET mental model

| .NET `PriorityQueue<TElement, TPriority>` (.NET 6+) | Python `heapq` on a `list` |
|---|---|
| `new PriorityQueue<string, int>()` | `heap = []` |
| `pq.Enqueue(item, priority)` | `heapq.heappush(heap, (priority, item))` |
| `pq.Dequeue()` | `heapq.heappop(heap)` |
| `pq.Peek()` | `heap[0]` |
| `pq.Count` | `len(heap)` |
| `pq.EnqueueDequeue(item, p)` | `heapq.heappushpop(heap, (p, item))` |
| `pq.DequeueEnqueue(item, p)` | `heapq.heapreplace(heap, (p, item))` |
| `new PriorityQueue<…>(items)` (O(n) build) | `heapq.heapify(list_)` |
| `items.OrderByDescending(f).Take(n)` | `heapq.nlargest(n, items, key=f)` |

Both are **min-heaps**: the smallest priority comes out first. The key difference is that Python stores the priority **inside** the item (usually as the first element of a tuple) rather than as a separate argument.

## The heap invariant

A heap is a list where `heap[k] <= heap[2*k+1]` and `heap[k] <= heap[2*k+2]` for every `k`. The consequence you care about: **`heap[0]` is always the smallest item**. The rest of the list is *not* sorted.

`heapify` turns an existing list into a heap **in place**, in O(n):

In [ ]:
import heapq

nums = [5, 1, 8, 3, 9, 2]
heapq.heapify(nums)
print(nums)          # heap order — not sorted!
print(nums[0])       # but the smallest is always first

## Push and pop

`heappush` and `heappop` are O(log n), just like `Enqueue` / `Dequeue`.

```csharp
// C#
var pq = new PriorityQueue<string, int>();
pq.Enqueue("write report", 2);
pq.Enqueue("fix outage", 0);
pq.Enqueue("lunch", 5);
while (pq.TryDequeue(out var task, out var priority))
    Console.WriteLine($"{priority}: {task}");
```

In [ ]:
tasks = []
heapq.heappush(tasks, (2, "write report"))
heapq.heappush(tasks, (0, "fix outage"))
heapq.heappush(tasks, (5, "lunch"))

print("peek:", tasks[0])
while tasks:
    priority, task = heapq.heappop(tasks)
    print(f"{priority}: {task}")

Tuples compare **element by element**, so `(priority, item)` sorts by priority first. That's why the tuple layout is the idiomatic replacement for .NET's separate `TPriority` argument.

⚠️ **Gotcha:** popping from an empty heap raises `IndexError` — there is no `TryDequeue`. Check `if heap:` first.

In [ ]:
try:
    heapq.heappop([])
except IndexError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** `heapq` trusts you. The list is a plain list, so `append`, `sort`, `insert` or assigning to an index all work — and silently **break the heap invariant**. Only modify a heap through the `heapq` functions (or call `heapify` again).

## Tie-breaking: `(priority, counter, item)`

When two entries have the same priority, Python compares the **next** tuple element. If the items themselves aren't comparable (dicts, most custom objects), you get a `TypeError`. Even when they are comparable, the order among equal priorities is decided by the items, not by insertion order.

In [ ]:
heap = []
heapq.heappush(heap, (1, {"order": 17}))
try:
    heapq.heappush(heap, (1, {"order": 42}))    # same priority -> compares the dicts
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

The standard fix is a monotonically increasing **counter** as the second element. It guarantees the comparison never reaches the item, and gives **FIFO order among equal priorities** — a stability guarantee .NET's `PriorityQueue` doesn't make.

In [ ]:
import itertools

class TaskQueue:
    def __init__(self):
        self._heap = []
        self._counter = itertools.count()     # 0, 1, 2, ...

    def push(self, item, priority: int) -> None:
        heapq.heappush(self._heap, (priority, next(self._counter), item))

    def pop(self):
        priority, _, item = heapq.heappop(self._heap)
        return item

    def __len__(self) -> int:
        return len(self._heap)

q = TaskQueue()
q.push({"order": 17}, priority=1)
q.push({"order": 42}, priority=1)
q.push({"order": 3}, priority=0)
while q:
    print(q.pop())

**Tip:** a `@dataclass(order=True)` with the payload marked `field(compare=False)` is another tidy way to make heap entries *(see Syntax › Classes & OOP › 008 · Dataclasses)*.

In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)
class Job:
    priority: int
    seq: int
    payload: dict = field(compare=False)

jobs = [Job(2, 0, {"name": "backup"}), Job(1, 1, {"name": "deploy"})]
heapq.heapify(jobs)
print(heapq.heappop(jobs).payload)

## `heappushpop` and `heapreplace`

Both combine a push and a pop in a single, faster operation:

- `heappushpop(heap, x)` — push `x`, **then** pop the smallest (may return `x` itself). ≈ `EnqueueDequeue`.
- `heapreplace(heap, x)` — pop the smallest, **then** push `x` (returns the old smallest, even if `x` is smaller). ≈ `DequeueEnqueue`.

In [ ]:
heap = [1, 3, 5]
print(heapq.heappushpop(heap, 0), heap)   # 0 is smallest -> returned immediately
print(heapq.heapreplace(heap, 0), heap)   # 1 is popped first, then 0 pushed

A classic use is keeping the **top-k** items of a stream in a fixed-size min-heap:

In [ ]:
def top_k(stream, k):
    heap = []
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)     # drop the smallest of the current top-k
    return sorted(heap, reverse=True)

print(top_k([5, 1, 9, 3, 7, 8, 2], k=3))

## `nlargest` and `nsmallest`

For one-off top-N queries you don't need to manage a heap yourself. Both accept a `key=` function, like `sorted`.

```csharp
// C#
var top2 = products.OrderByDescending(p => p.Price).Take(2);
```

In [ ]:
products = [
    {"name": "keyboard", "price": 49.0},
    {"name": "monitor", "price": 189.0},
    {"name": "mouse", "price": 19.5},
    {"name": "laptop stand", "price": 35.0},
]

print([p["name"] for p in heapq.nlargest(2, products, key=lambda p: p["price"])])
print([p["name"] for p in heapq.nsmallest(2, products, key=lambda p: p["price"])])

**Tip:** which tool for "smallest/largest"?

| Need | Use |
|---|---|
| just the single min/max | `min()` / `max()` |
| a few items out of many | `heapq.nsmallest` / `nlargest` |
| most of the items | `sorted(...)[:n]` |

`heapq.merge(*sorted_iterables)` lazily merges already-sorted inputs into one sorted stream — useful for merging sorted files or log streams without loading everything:

In [ ]:
print(list(heapq.merge([1, 4, 7], [2, 3, 9], [5])))

## Max-heaps

### The classic trick: negate the priority

For numbers, push `-priority` and negate again when popping. This works in every Python version. (In C# you'd pass a reversed `IComparer<TPriority>`.)

```csharp
// C#
var maxPq = new PriorityQueue<string, int>(Comparer<int>.Create((a, b) => b.CompareTo(a)));
```

In [ ]:
max_heap = []
for score, player in [(70, "alice"), (95, "bob"), (82, "carol")]:
    heapq.heappush(max_heap, (-score, player))

neg_score, player = heapq.heappop(max_heap)
print(player, -neg_score)

### Python 3.14: built-in max-heap functions

Python 3.14 adds public max-heap counterparts: `heapify_max`, `heappush_max`, `heappop_max`, `heappushpop_max` and `heapreplace_max`. On a max-heap, `heap[0]` is the **largest** item.

In [ ]:
scores = [70, 95, 82, 60]
heapq.heapify_max(scores)
print(scores[0])                       # largest

heapq.heappush_max(scores, 99)
print(heapq.heappop_max(scores))
print(heapq.heappop_max(scores))
print(sorted(n for n in dir(heapq) if n.endswith("_max") and not n.startswith("_")))

⚠️ **Gotcha:** don't mix the two families on the same list — a list heapified with `heapify_max` must only be used with the `*_max` functions.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Create | `new PriorityQueue<T, P>()` | `heap = []` |
| Build from items | `new PriorityQueue<T, P>(items)` | `heapq.heapify(lst)` |
| Enqueue | `pq.Enqueue(item, p)` | `heapq.heappush(heap, (p, n, item))` |
| Dequeue | `pq.Dequeue()` | `heapq.heappop(heap)` |
| Peek | `pq.Peek()` | `heap[0]` |
| Push then pop | `pq.EnqueueDequeue(item, p)` | `heapq.heappushpop(heap, x)` |
| Pop then push | `pq.DequeueEnqueue(item, p)` | `heapq.heapreplace(heap, x)` |
| Top N | `OrderByDescending(f).Take(n)` | `heapq.nlargest(n, xs, key=f)` |
| Merge sorted streams | hand-written | `heapq.merge(a, b, ...)` |
| Max-heap | reversed `IComparer` | negate priority, or `heappush_max` etc. (3.14) |

## Exercises

1. Use `heapq` to return the 3 cheapest products (by price) from a list of `(name, price)` tuples, without sorting the whole list.
2. Implement a simple event scheduler: given `(name, time)` pairs `("backup", 30)`, `("email", 10)`, `("report", 10)`, `("cleanup", 5)`, push them onto a heap and process them in time order, keeping insertion order for equal times.
3. Merge three sorted lists of sensor timestamps into one sorted list with `heapq.merge`.
4. Find the 2 highest scores using a max-heap (either technique).

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import heapq
import itertools

# 1
products = [("cable", 9.99), ("monitor", 189.0), ("mouse", 19.5), ("pad", 4.5), ("hub", 25.0)]
print(heapq.nsmallest(3, products, key=lambda p: p[1]))

# 2 — (time, counter, name) keeps FIFO order for equal times
counter = itertools.count()
events = []
for name, time in [("backup", 30), ("email", 10), ("report", 10), ("cleanup", 5)]:
    heapq.heappush(events, (time, next(counter), name))
while events:
    time, _, name = heapq.heappop(events)
    print(time, name)

# 3
a, b, c = [1, 5, 9], [2, 6], [3, 4, 10]
print(list(heapq.merge(a, b, c)))

# 4 — 3.14 max-heap functions (or push negated values with heappush)
scores = [70, 95, 82, 60, 88]
heapq.heapify_max(scores)
print([heapq.heappop_max(scores) for _ in range(2)])
```

</details>